In [2]:
import os
import json
import pandas as pd
from pathlib import Path


In [3]:
data_path = Path('/DATA/DATANAS2/rhh25/dti_dataset/')
#set current working directory to the library path
library_path = Path('/villa/rhh25/Cellhit/') 
os.chdir(library_path)

In [138]:
def merge_dicts(*dicts):
    result = {}
    for d in dicts:
        result.update(d)
    return result



In [5]:

with open("data/drug_smiles.json", "r") as f:
    smiles_drugs = json.load(f)

with open("data/drug_smiles1.json", "r") as f:
    smiles_drugs1 = json.load(f)

with open("data/drug_smiles2.json", "r") as f:
    smiles_drugs2 = json.load(f)

with open("data/drug_not_found.json", "r") as f:
    smiles_drugs_notfound = json.load(f)['not_found']

with open("data/drug_not_found1.json", "r") as f:
    smiles_drugs_notfound_1 = json.load(f)['not_found']

with open("data/drug_not_found2.json", "r") as f:
    smiles_drugs_notfound_2 = json.load(f)['not_found']

final_smile_drugs = merge_dicts(smiles_drugs, smiles_drugs1, smiles_drugs2)

drugcomb_smile_df = pd.DataFrame.from_dict(final_smile_drugs, orient='index').reset_index()

drugcomb_smile_df.columns = ['Drug Name', 'CID', 'Smiles']
drugcomb_smile_df.to_csv('/DATA/DATANAS2/rhh25/Cellhit/data/drugs/drugcomb_smile.csv', index=False)

In [6]:
with open("drug_mapped.json", "r") as f:
    drug_inchikey = json.load(f)

# Convert to DataFrame
drugcomb_inchiKey_df = pd.DataFrame(list(drug_inchikey.items()), columns=["Drug Name", "InchiKey"])
drugcomb_inchiKey_df.to_csv('/DATA/DATANAS2/rhh25/Cellhit/data/drugs/drugcomb_inchiKey.csv', index=False)

In [7]:
drugcomb_df = pd.read_csv('/DATA/DATANAS2/rhh25/Cellhit/data/metadata/drugcombs_scored.csv')


In [8]:
drugcomb_df = drugcomb_df.dropna(subset=['Drug1', 'Drug2'] )
unique_drugcomb_drugs =  set(drugcomb_df.Drug1.unique().tolist()).union(
        set(drugcomb_df.Drug2.unique().tolist())
    )

In [9]:
with open("drug_mapped.json", "r") as f:
    inchi_drug_mapped = json.load(f)

inchi_drugcomb = pd.DataFrame.from_dict(inchi_drug_mapped, orient='index').reset_index()
inchi_drugcomb.columns = ['Drug Name', 'InchiKey']
inchi_drugcomb

,Drug Name,InchiKey
0,XYLOMETAZOLINE HCL,YGWFCQYETHJKNX-UHFFFAOYSA-N
1,BAICALEIN,FXNFHKRTJBSTCS-UHFFFAOYSA-N
2,CHEMBL3348974,TTZHDVOVKQGIBA-SWMKZBNESA-N
3,AC-261066,HSAOETBFVAWNRP-UHFFFAOYSA-N
4,SILDENAFIL,BNRNXUUZRGQAQC-UHFFFAOYSA-N
...,...,...
6101,PF-04691502,XDLYKKIQACFMJG-UHFFFAOYSA-N
6102,PDK1 inhibitor,GCWCGSPBENFEPE-VWLOTQADSA-N
6103,Triamcinolone acetonide,YNDXUCZADRHECN-JNQJZLCISA-N
6104,Phenylephrine hydrochloride,OCYSGIYOVXAGKQ-FVGYRXGTSA-N


In [10]:
#df_drugcomb = pd.read_csv(data_path/'biosnap/ChG-DrugComb_drugs.csv')

## OOD_dataset

In [7]:
import json
import os
import glob
import pandas as pd

from pathlib import Path


#set current working directory to the library path
library_path = Path('/villa/rhh25/Cellhit/') 
os.chdir(library_path)

In [8]:
def get_dataframe(data):
    rows = []
    for key in ['train', 'ood_val', 'ood_test', 'iid_val', 'iid_test']:
        for entry in data[key]:
            rows.append({
                'smiles': entry['smiles'],
                'reg_label': entry['reg_label'],
                'assay_id': entry['assay_id'],
                'protein': entry['protein'],
                'cls_label': entry['cls_label'],
                'domain_id': entry['domain_id']
            })

    # Create DataFrame
    df = pd.DataFrame(rows)
    return df


pattern = '*core*ec50*sbap*.json'
files = glob.glob(pattern)

directory = '/DATA/DATANAS2/rhh25/dti_dataset/drugood_all/'
all_json_files = glob.glob(os.path.join(directory, '*.json'))
keywords = ['core', 'ec50', 'sbap']
matching_files = [f for f in all_json_files if all(k in f for k in keywords)]



with open(matching_files[0], "r") as f:
    core_ec50_assay = json.load(f)
with open(matching_files[1], "r") as f:
    core_ec50_protein = json.load(f)
with open(matching_files[2], "r") as f:
    core_ec50_protein_fam = json.load(f)
with open(matching_files[3], "r") as f:
    core_ec50_scaffold = json.load(f)
with open(matching_files[4], "r") as f:
    core_ec50_size = json.load(f)

assay_df = get_dataframe(core_ec50_assay['split'])
protein_df = get_dataframe(core_ec50_protein['split'])
protein_fam_df = get_dataframe(core_ec50_protein_fam['split'])
scaffold_df = get_dataframe(core_ec50_scaffold['split'])
size_df = get_dataframe(core_ec50_size['split'])

over_all_df = pd.concat([assay_df, protein_df, protein_fam_df, scaffold_df, size_df], ignore_index=True)


### SMiles to CID mapping

In [ ]:
cid_path = '/villa/rhh25/Cellhit/data/cid_smile_mapping.json'

with open(cid_path, "r") as f:
    ood_cid = json.load(f)


unique_drug = over_all_df.smiles.unique()

ood_cid_df = pd.DataFrame.from_dict(ood_cid, orient='index').stack().reset_index()
ood_cid_df = ood_cid_df.rename(columns={'level_0': 'smiles', 0: 'cid'})
ood_cid_df = ood_cid_df.drop(columns=['level_1'])
ood_filer = over_all_df.merge(ood_cid_df, on='smiles', how='inner')
ood_cid_lst = ood_filer['cid'].unique().tolist()

In [ ]:
num_drugcomb = drug_comb_smile_df[drug_comb_smile_df['CID'].isin(ood_cid_lst)].shape[0]
print(f"Number of drugcomb after filtering with ood cid list: {num_drugcomb}")

### Inchi Key to CID mapping

In [10]:
assay_df

,smiles,reg_label,assay_id,protein,cls_label,domain_id
0,N#Cc1cc(NC(=O)c2ccc(F)cc2)c(O)n2c1nc1ccccc12,4.46,688113,MKAASNSVSSAGGSVSPTTTQPPLPPGQSSHPQIYDQQMQYYFAAA...,0,55
1,O=C1NC(=O)N(Cc2ccccc2)C(=O)/C1=C/NNC(=O)c1ccncc1,4.03,688113,MKAASNSVSSAGGSVSPTTTQPPLPPGQSSHPQIYDQQMQYYFAAA...,0,55
2,O=C1CSC(=S)N1/N=C/c1ccc(Cl)c([N+](=O)[O-])c1,4.80,688113,MKAASNSVSSAGGSVSPTTTQPPLPPGQSSHPQIYDQQMQYYFAAA...,0,55
3,Cc1ccccc1N1C(=O)/C(=C/NNC(=O)c2ccccc2O)C(=O)NC1=S,4.63,688113,MKAASNSVSSAGGSVSPTTTQPPLPPGQSSHPQIYDQQMQYYFAAA...,0,55
4,CC1=Nc2ccccc2/C1=C\c1nc(-c2ccc(F)cc2)oc1O,5.26,688113,MKAASNSVSSAGGSVSPTTTQPPLPPGQSSHPQIYDQQMQYYFAAA...,1,55
...,...,...,...,...,...,...
17763,COc1cc(N)c(Cl)cc1C(=O)NC1CCN(CCN2CCCCC2)CC1,7.00,583012,MDKLDANVSSEEGFGSVEKVVLLTFLSTVILMAILGNLLVMVAVCW...,1,32
17764,CC(C)n1c(=O)c(C(=O)N[C@@H]2C[C@@H]3CC[C@H](C2)...,8.10,583012,MDKLDANVSSEEGFGSVEKVVLLTFLSTVILMAILGNLLVMVAVCW...,1,32
17765,CC(C)n1nc(C(=O)NC2CCNCC2)c2ccccc21,6.70,583012,MDKLDANVSSEEGFGSVEKVVLLTFLSTVILMAILGNLLVMVAVCW...,1,32
17766,COc1cc(N)c(Cl)cc1C(=O)NC1CCN(CCCCCN2CCCCC2)CC1,8.30,583012,MDKLDANVSSEEGFGSVEKVVLLTFLSTVILMAILGNLLVMVAVCW...,1,32


## BidingDB

In [11]:
import json
import os
import glob
import pandas as pd

from pathlib import Path


#set current working directory to the library path
library_path = Path('/villa/rhh25/Cellhit/') 
os.chdir(library_path)

data_path = Path('/DATA/DATANAS2/rhh25/dti_dataset/')

In [12]:
df = pd.read_csv(data_path / 'bindingdb/BindingDB_All.tsv', sep='\t')

/tmp/ipykernel_2298505/918526288.py:1: DtypeWarning: Columns (2,3,7,8,9,10,11,12,13,15,17,18,20,21,22,25,29,30,34,35,37,38,41,42,43,44,45,47,48,49,50,52,53,54,55,56,57,59,60,61,62,64,65,66,67,68,69,71,72,73,74,76,77,78,79,80,81,83,84,85,86,88,89,90,91,92,93,95,96,97,100,101,102,103,104,105,107,108,109,112,113,114,115,116,117,119,120,121,124,125,126,127,128,129,131,132,133,136,137,138,139,140,141,143,144,145,148,149,150,151,152,153,155,156,157,160,161,162,163,164,165,167,168,169,172,173,174,175,176,177,179,180,181,184,185,186,187,188,189,191,192,193,196,197,198,199,200,201,203,204,205,208,209,210,211,212,213,215,216,217,220,221,222,223,224,225,227,228,229,232,233,234,235,236,237,239,240,241,244,245,246,247,248,249,251,252,253,256,257,258,259,260,261,263,264,265,268,269,270,271,272,273,275,276,277,280,281,282,283,284,285,287,288,289,292,293,294,295,296,297,299,300,301,304,305,306,307,308,309,311,312,313,316,317,318,319,320,321,328,329,330,331,332,333,340,341,342,343,344,345,352,353,354,3

In [13]:
#df_not_nan = df.dropna(subset=['Ligand SMILES', 'Target Name', 'EC50 (nM)'])

### Mapping with CID

In [14]:
active_binding_lst = df['BindingDB Reactant_set_id'].unique().tolist()

In [15]:
mapping_df = pd.read_csv(data_path / 'bindingdb/BindingDB_CID.txt', names=["bindingdb_id", "chembl_cid"], sep="\t")  # or sep=","
mapping_df = mapping_df[mapping_df['bindingdb_id'].isin(active_binding_lst)]
mapping_df.columns = ['bindingdb_id', 'CID']
chembl_cid_lst = mapping_df['CID'].tolist()


drugcomb_cid_bindingdb_df = pd.merge(drugcomb_smile_df, mapping_df, on='CID', how='inner')
drugcomb_cid_bindingdb_df.rename(columns={'bindingdb_id': 'BindingDB Reactant_set_id'}, inplace=True)

#drugcomb_cid_bindingdb_df = drugcomb_smile_df[drugcomb_smile_df['CID'].isin(chembl_cid_lst)]


In [16]:
cid_mapped_drugs = drugcomb_smile_df[drugcomb_smile_df['CID'].isin(chembl_cid_lst)]['Drug Name'].tolist()
print(f"Number of drugcomb after filtering with chembl cid list: {len(cid_mapped_drugs)}")

Number of drugcomb after filtering with chembl cid list: 3937


### Mapping with InchiKey

In [17]:
bindingId_ligand_df = df[['BindingDB Reactant_set_id', 'Ligand InChI Key']]
inchi_drugcomb.rename(columns={'InchiKey': 'Ligand InChI Key'}, inplace=True)
drugcomb_ligand_bindingdb_df = pd.merge(inchi_drugcomb, bindingId_ligand_df, on='Ligand InChI Key', how='inner')

In [18]:
inchi_key_lst = df['Ligand InChI Key'].dropna().unique().tolist()
inchi_mapped_drugs = inchi_drugcomb[inchi_drugcomb['Ligand InChI Key'].isin(inchi_key_lst)]['Drug Name'].tolist()

### All

In [19]:
bindingDB_drugcomb_drugs = set(cid_mapped_drugs).union(set(inchi_mapped_drugs))

binding_drugcomb_df =  drugcomb_df[drugcomb_df['Drug1'].isin(bindingDB_drugcomb_drugs) & drugcomb_df['Drug2'].isin(bindingDB_drugcomb_drugs)]
fullfill_percent = binding_drugcomb_df.shape[0]/ drugcomb_df.shape[0]
drugcomb_bindingdb_df = pd.concat([drugcomb_ligand_bindingdb_df[['Drug Name', 'BindingDB Reactant_set_id']], drugcomb_cid_bindingdb_df[['Drug Name', 'BindingDB Reactant_set_id']]], ignore_index=True).drop_duplicates()
active_binding_id = drugcomb_bindingdb_df['BindingDB Reactant_set_id'].unique().tolist()
active_biniding_df = df[df['BindingDB Reactant_set_id'].isin(active_binding_id)]
active_drugcomb_binding_df = drugcomb_bindingdb_df.merge(active_biniding_df, on='BindingDB Reactant_set_id', how='inner')[['Drug Name', 'Target Name', 'EC50 (nM)']].drop_duplicates()

print("Number of drugs in drugcomb mapped to bindingDB: ", len(bindingDB_drugcomb_drugs))
print(f"Number of drugcomb after filtering with bindingDB mapped drugs: {binding_drugcomb_df.shape[0]}")
print(f"Percentage of drugcomb after filtering with bindingDB mapped drugs: {fullfill_percent:.2%}")
print("Number of unique targets in active drugcomb binding data: ", active_drugcomb_binding_df['Target Name'].nunique())



Number of drugs in drugcomb mapped to bindingDB:  4187
Number of drugcomb after filtering with bindingDB mapped drugs: 272262
Percentage of drugcomb after filtering with bindingDB mapped drugs: 54.58%
Number of unique targets in active drugcomb binding data:  2905


In [61]:
active_drugcomb_binding_df['Target Name'].nunique()

2905

In [56]:
active_drugcomb_binding_df

,Drug Name,Target Name,EC50 (nM)
0,BAICALEIN,17-beta-hydroxysteroid dehydrogenase type 3,NaN
1,BAICALEIN,Linoleate 9S-lipoxygenase-4,NaN
2,BAICALEIN,Polyunsaturated fatty acid lipoxygenase ALOX12,NaN
3,BAICALEIN,Polyunsaturated fatty acid lipoxygenase ALOX15,NaN
4,BAICALEIN,Polyunsaturated fatty acid lipoxygenase ALOX15B,NaN
...,...,...,...
166722,MEROPENEM,Corticotropin-releasing factor receptor 1,NaN
166723,CHLOROQUINE,Coagulation factor X,NaN
166724,Ingenol 3-angelate,Stromelysin-1,NaN
166725,NALIDIXIC ACID,Cyclin-H/Cyclin-dependent kinase 7,NaN


### Protein

In [118]:
from services.genes import get_active_genes

In [119]:
celline_data_path = '/DATA/DATANAS2/rhh25/Cellhit/data/'
protein_path = '/DATA/DATANAS2/rhh25/ppi/'


ccle_expression = pd.read_csv(f'{celline_data_path}/ccle/OmicsExpressionEffectiveLengthHumanAllGenesStranded.csv', index_col=0)
df_drugcombs = pd.read_csv(f'{celline_data_path}/metadata/drugcombs_scored.csv')
metadata = pd.read_csv(f"{celline_data_path}/ccle/Model.csv")

with open(protein_path + 'protein_to_gene_mapping.json', "r") as f:
    protein_to_genes = json.load(f)

protein_gene_df = pd.DataFrame(list(protein_to_genes.items()), columns=["protein", "gene"])


In [120]:
protein_gene_df

,protein,gene
0,9606.ENSP00000358072,ATG5
1,9606.ENSP00000296694,SCGB3A2
2,9606.ENSP00000360060,FRAT1
3,9606.ENSP00000499042,SMIM48
4,9606.ENSP00000240423,NCAPH
...,...,...
18958,9606.ENSP00000221459,LIN7B
18959,9606.ENSP00000253083,HIP1R
18960,9606.ENSP00000485287,OR2L8
18961,9606.ENSP00000359866,BMP5


In [123]:
active_genes, ccle_active_expression, df_modelid_drugcombs = get_active_genes(df_drugcombs, metadata, ccle_expression, threshold=1000)

active_protein_gene_df = protein_gene_df[protein_gene_df['gene'].isin(active_genes)]
active_protein_lst = active_protein_gene_df['protein'].unique().tolist()
len(active_protein_lst)

Number of unique cell lines in DrugComb: 123
Gene expression columns: 60649
Number of active genes: 23968
Number of cell lines with active genes:  94


15062

In [ ]:
ppi_file_path = '/DATA/DATANAS2/rhh25/ppi/9606.protein.links.full.v12.0.txt' 
print(f"Loading PPI data from {ppi_file_path}...")

# Read the PPI file
# Format: protein1 \t protein2 \t combined_score
ppi_data = pd.read_csv(ppi_file_path, sep=' ', header=None,  skiprows=1,
                        names=['protein1', 'protein2', 'neighborhood', 'neighborhood_transferred', 'fusion', 'cooccurence', 'homology', 'coexpression', 'coexpression_transferred', 'experiments', 'experiments_transferred', 'database', 'database_transferred', 'textmining', 'textmining_transferred', 'combined_score'])

all_proteins = list(set(ppi_data['protein1'].tolist()).union(set(ppi_data['protein2'].tolist())))

## Get active PPI data

protein_gene_df = pd.DataFrame(list(protein_to_genes.items()), columns=["protein", "gene"])
active_protein_gene_df = protein_gene_df[protein_gene_df['gene'].isin(active_genes)]
active_proteins = active_protein_gene_df['protein'].unique()
active_ppi_data = ppi_data[ppi_data['protein1'].isin(active_proteins) & ppi_data['protein2'].isin(active_proteins)]

Loading PPI data from /DATA/DATANAS2/rhh25/ppi/9606.protein.links.full.v12.0.txt...


In [140]:
## Get active PPI data

protein_gene_df = pd.DataFrame(list(protein_to_genes.items()), columns=["protein", "gene"])
active_protein_gene_df = protein_gene_df[protein_gene_df['gene'].isin(active_genes)]
active_proteins = active_protein_gene_df['protein'].unique()
active_ppi_data = ppi_data[ppi_data['protein1'].isin(active_proteins) & ppi_data['protein2'].isin(active_proteins)]

In [ ]:
active_ppi_data.to_csv('/DATA/DATANAS2/rhh25/Cellhit/data/graph_data_2/active_ppi_data.csv', index=False)


In [134]:
df_modelid_drugcombs.dropna(subset=['ModelID'], inplace=True)

In [ ]:
# df_modelid_drugcombs[['Cell line', 'ModelID']].to_csv(f'{celline_data_path}/metadata/drugcombs_modelid.csv', index=False)

In [ ]:
#ccle_active_expression.to_csv('/DATA/DATANAS2/rhh25/Cellhit/data/graph_data_2/cell_line_expression.csv', index=True)

In [44]:
ccle_active_expression

,TSPAN6,SCYL3,FIRRM,FGR,CFH,FUCA2,GCLC,NFYA,STPG1,NIPAL3,...,UGT1A3,ENSG00000288703,UGT1A5,F8A2,ENSG00000288710,ENSG00000288719,ENSG00000288720,ENSG00000288721,F8A1,ENSG00000288725
ModelID,,,,,,,,,,,,,,,,,,,,,
ACH-000421,2243.94,3161.30,1701.90,1975.88,1976.21,1815.87,1837.67,3863.15,2558.28,3895.77,...,2206.02,3382.00,2213.84,1079.60,1267.26,3969.64,3001.67,4348.41,1071.54,3465.78
ACH-000754,2612.80,3096.27,1468.35,2421.73,1389.16,1844.98,1531.34,3155.06,2172.43,3429.34,...,2132.18,3396.00,2149.06,1450.68,1308.27,3280.98,1880.52,1581.74,1431.23,2859.44
ACH-000376,2337.36,3238.92,1253.55,2008.33,2002.72,1839.76,1697.77,3842.33,1656.17,4364.15,...,2083.51,3369.00,2100.05,1403.62,1230.46,3321.45,1942.87,2294.99,1389.01,2861.89
ACH-000035,2313.40,3423.39,1663.35,2143.41,3104.54,1767.15,1840.00,3816.07,1963.47,3367.17,...,2022.73,2903.51,2035.22,1155.67,1186.57,3477.70,2036.75,1782.35,1141.94,2991.48
ACH-000322,2222.47,3461.79,1533.21,1045.33,1424.01,1850.42,1943.40,3902.62,1954.27,4221.59,...,2120.49,3370.00,2130.62,1162.06,1786.49,3499.69,1951.54,1738.14,1148.77,3001.67
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ACH-000900,2282.33,3154.42,1507.39,2315.90,2005.01,1859.98,1985.78,4209.64,1981.44,3647.20,...,2112.10,3365.00,2126.40,1247.08,1354.39,3486.30,2123.05,1182.44,1232.58,2983.32
ACH-000115,2459.37,3137.34,1454.32,2231.76,1964.33,1840.83,1868.33,3578.41,2016.31,4019.15,...,2134.19,3357.00,2150.66,1364.68,1406.58,3449.65,1967.95,1688.43,1350.11,2963.16
ACH-000046,2445.02,3456.65,1500.17,2107.42,2474.50,1871.91,1863.63,4247.72,2106.06,3433.26,...,2135.75,3371.00,2153.03,1060.91,1574.22,3605.22,2050.85,1805.14,1046.69,3079.06


In [45]:
mapping_bindingdb_to_ppi_df = pd.read_csv('bindingdb_protein_mapping_results.csv')
active_drugcomb_binding_string_df = pd.merge(mapping_bindingdb_to_ppi_df, active_drugcomb_binding_df, on='Target Name', how='inner')
print("Number of unique drugs in active drugcomb binding string data: ", active_drugcomb_binding_string_df['Drug Name'].nunique())#full_active_protein = mapping_bindingdb_to_ppi_df['Target Name'].unique().tolist()

active_people_drugcomb_binding_string_df = active_drugcomb_binding_string_df[active_drugcomb_binding_string_df['STRING_id'].isin(active_protein_lst)]

print("Number of unique drugs in active drugcomb binding string data after filtering with active protein list: ", active_people_drugcomb_binding_string_df['Drug Name'].nunique())
active_people_drugcomb_binding_string_df.rename(columns={'STRING_id': 'Protein'}, inplace=True)
drugcomb_target_binding_df = active_people_drugcomb_binding_string_df[['Drug Name', 'Protein']].drop_duplicates().reset_index(drop=True)

#active_people_drugcomb_binding_string_df = active_people_drugcomb_binding_string_df[['Drug Name', 'STRING_id']].drop_duplicates()


Number of unique drugs in active drugcomb binding string data:  4070
Number of unique drugs in active drugcomb binding string data after filtering with active protein list:  3747


/tmp/ipykernel_2298505/239166894.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  active_people_drugcomb_binding_string_df.rename(columns={'STRING_id': 'Protein'}, inplace=True)


In [65]:
drugcomb_target_binding_df.Protein.nunique()

1495

In [82]:
drugcomb_target_binding_df

,Drug Name,Protein
0,BAICALEIN,9606.ENSP00000354612
1,Celecoxib,9606.ENSP00000354612
2,SULFASALAZINE,9606.ENSP00000354612
3,PRANOPROFEN,9606.ENSP00000354612
4,ROFECOXIB,9606.ENSP00000354612
...,...,...
52927,prochlorperazine,9606.ENSP00000350198
52928,GNF-6231,9606.ENSP00000259008
52929,NVP-ABE171,9606.ENSP00000444271
52930,Citalopram hydrobromide,9606.ENSP00000444810


## Biosnap

In [28]:

targetDeca_df = pd.read_csv(data_path/'biosnap/ChG-TargetDecagon_targets.csv')
targetDeca_df['# Drug\tGene'] = targetDeca_df['# Drug\tGene'].apply(lambda x: str(x))
targetDeca_df = targetDeca_df['# Drug\tGene'].str.split('\t', expand=True).reset_index()
targetDeca_df.columns = ['Drug', 'Gene']


## With DrugBank drugs
miner_chem_df = pd.read_csv(data_path/'biosnap/ChG-Miner_miner-chem-gene.tsv')
miner_chem_df[['Drug', 'Gene']] = miner_chem_df['#Drug\tGene'].str.split('\t', expand=True)
miner_chem_df.drop(columns=['#Drug\tGene'], inplace=True)
miner_chem_df.columns = ['DrugBankId', 'UniprotGene']
drugbank_lst = miner_chem_df['DrugBankId'].unique().tolist()


syn_df = pd.read_csv(data_path/'biosnap/CID-Synonym-filtered', sep="\t", header=None)
syn_df.columns = ['CID', 'DrugBankId']
syn_df_filter = syn_df[syn_df['DrugBankId'].isin(drugbank_lst)]


biosnap_df = pd.merge(syn_df_filter, drugcomb_smile_df, on='CID', how='inner')
biosnap_df = pd.merge(biosnap_df, miner_chem_df, on='DrugBankId', how='inner')
biosnap_drugs = biosnap_df['Drug Name'].unique().tolist()

## mapping to gene symbol
uniprot_to_gene_symbol_mapping_df = pd.read_csv(data_path/'biosnap/uniprot_to_gene_symbol_mapping.csv')
uniprot_to_gene_symbol_mapping_df.rename(columns={'UniProt': 'UniprotGene'}, inplace=True)
biosnap_gene_symbol_df = pd.merge(biosnap_df, uniprot_to_gene_symbol_mapping_df, on='UniprotGene', how='inner')

In [29]:
biosnap_gene_symbol_df

,CID,DrugBankId,Drug Name,Smiles,UniprotGene,Gene
0,237,DB01103,Quinacrine,CCN(CC)CCCC(C)NC1=C2C=C(C=CC2=NC3=C1C=CC(=C3)C...,O60733,PLA2G6
1,237,DB01103,Quinacrine,CCN(CC)CCCC(C)NC1=C2C=C(C=CC2=NC3=C1C=CC(=C3)C...,P20815,CYP3A5
2,237,DB01103,Quinacrine,CCN(CC)CCCC(C)NC1=C2C=C(C=CC2=NC3=C1C=CC(=C3)C...,Q15111,PLCL1
3,237,DB01103,Quinacrine,CCN(CC)CCCC(C)NC1=C2C=C(C=CC2=NC3=C1C=CC(=C3)C...,P47712,PLA2G4A
4,237,DB01103,Quinacrine,CCN(CC)CCCC(C)NC1=C2C=C(C=CC2=NC3=C1C=CC(=C3)C...,P08684,CYP3A4
...,...,...,...,...,...,...
8811,135565674,DB06695,211915-06-9,CCCCCCOC(=O)NC(=N)C1=CC=C(C=C1)NCC2=NC3=C(N2C)...,O00748,CES2
8812,135565674,DB06695,211915-06-9,CCCCCCOC(=O)NC(=N)C1=CC=C(C=C1)NCC2=NC3=C(N2C)...,P16083,NQO2
8813,135565674,DB06695,211915-06-9,CCCCCCOC(=O)NC(=N)C1=CC=C(C=C1)NCC2=NC3=C(N2C)...,P16662,UGT2B7
8814,135565674,DB06695,211915-06-9,CCCCCCOC(=O)NC(=N)C1=CC=C(C=C1)NCC2=NC3=C(N2C)...,P23141,CES1


### Protein

In [30]:
protein_gene_df

,protein,gene
0,9606.ENSP00000358072,ATG5
1,9606.ENSP00000296694,SCGB3A2
2,9606.ENSP00000360060,FRAT1
3,9606.ENSP00000499042,SMIM48
4,9606.ENSP00000240423,NCAPH
...,...,...
18958,9606.ENSP00000221459,LIN7B
18959,9606.ENSP00000253083,HIP1R
18960,9606.ENSP00000485287,OR2L8
18961,9606.ENSP00000359866,BMP5


In [31]:
protein_gene_df.rename(columns={'gene': 'Gene', 'protein':'Protein'}, inplace=True)
biosnap_gene_symbol_string_df = pd.merge(biosnap_gene_symbol_df, protein_gene_df, on='Gene', how='inner')
drugcomb_target_biosnap_df = biosnap_gene_symbol_string_df[['Drug Name', 'Protein']].drop_duplicates().reset_index(drop=True)


In [32]:
combine_drugs = set(bindingDB_drugcomb_drugs).union(set(biosnap_drugs))

print(f"Number of unique drugs in bindingDB and biosnap mapped drugcomb: {len(combine_drugs)}")
filtered_drugcomb_df =  drugcomb_df[drugcomb_df['Drug1'].isin(combine_drugs) & drugcomb_df['Drug2'].isin(combine_drugs)]
fullfill_percent = filtered_drugcomb_df.drop_duplicates(subset=['Drug1', 'Drug2', 'Cell line']).shape[0]/ drugcomb_df.drop_duplicates(subset=['Drug1', 'Drug2', 'Cell line']).shape[0]
print(f"Number of drugcomb after filtering with bindingDB, BioSnap mapped drugs: {filtered_drugcomb_df.shape[0]}")
print(f"Percentage of drugcomb after filtering with bindingDB, BioSnap mapped drugs: {fullfill_percent:.2%}")



Number of unique drugs in bindingDB and biosnap mapped drugcomb: 4295
Number of drugcomb after filtering with bindingDB, BioSnap mapped drugs: 301277
Percentage of drugcomb after filtering with bindingDB, BioSnap mapped drugs: 58.20%


In [33]:
drugcomb_target_df = pd.concat([drugcomb_target_binding_df, drugcomb_target_biosnap_df], ignore_index=True).drop_duplicates()
drugcomb_target_drugs = drugcomb_target_df['Drug Name'].unique().tolist()
print("Number of unique drug-target interactions in drugcomb target df: ", drugcomb_target_df.shape[0])
print("Number of unique drugs in drugcomb target df: ", len(drugcomb_target_drugs))

protein_filtered_drugcomb_df =  drugcomb_df[drugcomb_df['Drug1'].isin(drugcomb_target_drugs) & drugcomb_df['Drug2'].isin(drugcomb_target_drugs)]
fullfill_percent = protein_filtered_drugcomb_df.drop_duplicates(subset=['Drug1', 'Drug2','Cell line']).shape[0]/ drugcomb_df.drop_duplicates(subset=['Drug1', 'Drug2','Cell line']).shape[0]
print(f"Number of drugcomb after filtering with bindingDB, BioSnap mapped drugs and String proteins: {protein_filtered_drugcomb_df.shape[0]}")
print(f"Percentage of drugcomb after filtering with bindingDB, BioSnap mapped drugs and String proteins: {fullfill_percent:.2%}")




Number of unique drug-target interactions in drugcomb target df:  60027
Number of unique drugs in drugcomb target df:  3974


Number of drugcomb after filtering with bindingDB, BioSnap mapped drugs and String proteins: 292348
Percentage of drugcomb after filtering with bindingDB, BioSnap mapped drugs and String proteins: 56.19%


In [92]:
protein_gene_df.rename(columns={'gene': 'Gene', 'protein':'Protein'}, inplace=True)

In [94]:
drugcomb_target_df =drugcomb_target_df.merge(protein_gene_df, on='Protein', how='inner')

In [105]:
drugcomb_target_df

,Drug Name,Protein,Gene
0,BAICALEIN,9606.ENSP00000354612,PTGS1
1,Celecoxib,9606.ENSP00000354612,PTGS1
2,SULFASALAZINE,9606.ENSP00000354612,PTGS1
3,PRANOPROFEN,9606.ENSP00000354612,PTGS1
4,ROFECOXIB,9606.ENSP00000354612,PTGS1
...,...,...,...
60022,211915-06-9,9606.ENSP00000341045,UGT2B15
60023,211915-06-9,9606.ENSP00000346768,UGT1A9
60024,211915-06-9,9606.ENSP00000369822,NQO2
60025,211915-06-9,9606.ENSP00000304811,UGT2B7


In [34]:
cellLine_modelID_lst = ccle_active_expression.reset_index().ModelID.unique().tolist()


df_drugcomb_cellLine_filter_df = df_drugcombs[df_drugcombs['ModelID'].isin(cellLine_modelID_lst)]
df_drugcomb_cellLine_filter_df.drop_duplicates(subset=['Drug1', 'Drug2', 'CellLineClean'], inplace=True)

cellline_filtered_drugcomb_df =  df_drugcomb_cellLine_filter_df[df_drugcomb_cellLine_filter_df['Drug1'].isin(drugcomb_target_drugs) & df_drugcomb_cellLine_filter_df['Drug2'].isin(drugcomb_target_drugs)]
fullfill_percent = cellline_filtered_drugcomb_df.drop_duplicates(subset=['Drug1', 'Drug2','Cell line']).shape[0]/ df_drugcomb_cellLine_filter_df.drop_duplicates(subset=['Drug1', 'Drug2','Cell line']).shape[0]

celline_drug_number = len(set(cellline_filtered_drugcomb_df['Drug1'].unique().tolist()).union(set(cellline_filtered_drugcomb_df['Drug2'].unique().tolist())))
print(f"Number of unique drugs in drugcomb after filtering with cell line list: {celline_drug_number}")
print(f"Number of drugcomb after filtering with bindingDB, BioSnap mapped drugs and String proteins: {cellline_filtered_drugcomb_df.shape[0]}")
print(f"Percentage of drugcomb after filtering with bindingDB, BioSnap mapped drugs and String proteins: {fullfill_percent:.2%}")

Number of unique drugs in drugcomb after filtering with cell line list: 2969
Number of drugcomb after filtering with bindingDB, BioSnap mapped drugs and String proteins: 160450
Percentage of drugcomb after filtering with bindingDB, BioSnap mapped drugs and String proteins: 54.54%


/tmp/ipykernel_2298505/3808764819.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_drugcomb_cellLine_filter_df.drop_duplicates(subset=['Drug1', 'Drug2', 'CellLineClean'], inplace=True)


In [35]:
drugcomb_target_df['Protein'].nunique()

1948

In [107]:
len(cellLine_modelID_lst)

93

In [37]:
df_drugcombs[df_drugcombs['ModelID'].isin(cellLine_modelID_lst)]['Drug1'].nunique()

4715

In [38]:
data_path

PosixPath('/DATA/DATANAS2/rhh25/dti_dataset')

In [ ]:
#drugcomb_target_df.to_csv(str(data_path)+'/drugcomb_target/drugcomb_target_df.csv', index=False)

## Cell Line & Drug Bank

In [ ]:
drugbank_path = '/DATA/DATANAS2/rhh25/dti_dataset/drugbank/'
drugbank_df = pd.read_csv(drugbank_path + 'database.csv') 

In [154]:
drug_name_drug_banks = drugbank_df['Common name'].dropna().unique().tolist()
len(drug_name_drug_banks)

17420

In [159]:
df_drugcombs

,ID,Drug1,Drug2,Cell line,ZIP,Bliss,Loewe,HSA,CellLineClean,ModelID
0,1,5-FU,ABT-888,A2058,1.720,6.260,-2.750,5.540,A2058,ACH-000788
1,2,5-FU,ABT-888,A2058,5.880,12.330,3.330,11.610,A2058,ACH-000788
2,3,5-FU,ABT-888,A2058,3.590,11.660,2.650,10.940,A2058,ACH-000788
3,4,5-FU,ABT-888,A2058,-0.850,5.150,-3.860,4.430,A2058,ACH-000788
4,5,5-FU,AZD1775,A2058,12.290,15.770,10.400,18.660,A2058,ACH-000788
...,...,...,...,...,...,...,...,...,...,...
498860,498861,Mitomycin C,Valproic acid sodium salt,DIPG25,-3.076,-11.124,-6.078,-6.367,DIPG25,None
498861,498862,Cyanein,Valproic acid sodium salt,DIPG25,1.320,-1.091,-16.157,1.139,DIPG25,None
498862,498863,Erlotinib,Valproic acid sodium salt,DIPG25,-15.768,-15.776,-25.465,-6.212,DIPG25,None
498863,498864,Bafilomycin A1,Valproic acid sodium salt,DIPG25,2.208,10.567,-9.455,1.525,DIPG25,None


## Kiba & Davis

In [163]:
## Kiba

In [45]:
kiba_df = pd.read_csv(data_path/'davis_kiba/kiba_all.csv')
kiba_drugs = kiba_df.compound_iso_smiles.unique().tolist()
kiba_df
#drug_comb_smile_df[drug_comb_smile_df['Smiles'].isin(kiba_drugs)]

,compound_iso_smiles,target_sequence,affinity
0,O=C1c2c(c3c4ccc(O)cc4n(C4OC(CO)C(O)C(O)C4O)c3c...,MSANNSPPSAQKSVLPTAIPAVLPAASPCSSPKTGLSARLSNGSFS...,9.798970
1,N#Cc1ccc(NC(=O)Nc2ccnc3cc(C(F)(F)F)ccc23)nc1,MLGAVEGPRWKQAEDIRDIYDFRDVLGTGAFSEVILAEDKRTQKLV...,11.400000
2,CCNc1nc2cc(Cl)c(OC)cc2nc1NCC,MEPAAGFLSPRPFQRAAAAPAPPAGPGPPPSALRGPELEMLAGLPT...,11.200000
3,CNc1ncnc2c1N=C(c1ccc(NC(=O)Nc3cccc(C(F)(F)F)c3...,MELRVGNRYRLGRKIGSGSFGDIYLGTDIAAGEEVAIKLECVKTKH...,11.200000
4,C#Cc1cc2c(cc1OC)-c1[nH]nc(-c3ccc(C#N)nc3)c1C2,MRGARGAWDFLCVLLLLLRVQTGSSQPSVSPGEPSPPSIHPGKSDL...,11.999998
...,...,...,...
118249,CN(C)c1cc2sncc2cc1NC(=O)C(=O)O,MSELEEDFAKILMLKEERIKELEKRLSEKEEEIQELKRKLHKCQSV...,11.800001
118250,CN1CCC1COc1cncc(CCc2ccncc2)c1,MAPFLRIAFNSYELGSLQAEDEANQPFCAVKMKEALSTERGKTLVQ...,11.200000
118251,O=C(CO)N1CCC(c2[nH]nc(-c3ccc(Cl)cc3F)c2-c2ccnc...,MSSWIRWHGPAMARLWGFCWLVVGFWRAAFACPTSCKCSASRIWCS...,11.400000
118252,NNc1cc(N2CCOCC2)nc(OCCc2ccccn2)n1,MATCIGEKIEDFKVGNLLGKGSFAGVYRAESIHTGLEVAIKMIDKK...,11.500000


In [182]:
len(set(kiba_drugs) - set(unique_drug) )

2064

In [183]:
kiba_drugs

['O=C1c2c(c3c4ccc(O)cc4n(C4OC(CO)C(O)C(O)C4O)c3c3[nH]c4cc(O)ccc4c23)C(=O)N1NC(CO)CO',
 'N#Cc1ccc(NC(=O)Nc2ccnc3cc(C(F)(F)F)ccc23)nc1',
 'CCNc1nc2cc(Cl)c(OC)cc2nc1NCC',
 'CNc1ncnc2c1N=C(c1ccc(NC(=O)Nc3cccc(C(F)(F)F)c3)cc1)CCN2',
 'C#Cc1cc2c(cc1OC)-c1[nH]nc(-c3ccc(C#N)nc3)c1C2',
 'COC1C(N(C)C(=O)c2cccc([N+](=O)[O-])c2)CC2OC1(C)n1c3ccccc3c3c4c(c5c6ccccc6n2c5c31)C(=O)NC4',
 'CCCC(=O)Nc1cccc(-c2nc(Nc3ccc4[nH]ncc4c3)c3cc(OCCN4CCCC4)ccc3n2)c1',
 'Cc1cccc(NC(=O)Nc2ccc(-c3csc4c(C#CCN5CCOCC5)cnc(N)c34)cc2)c1',
 'Cc1[nH]c(C=C2C(=O)Nc3ccccc32)c(C)c1CCC(=O)O',
 'Cc1nnc(-c2cc3c(Oc4ccc(C(F)(F)F)cc4)cncc3s2)o1',
 'N#Cc1c2c3cc(Cl)ccc3oc(N)c-2c(=O)[nH]c1=O',
 'Cc1ccc(C(=O)Nc2ccon2)cc1Nc1ncnc2c1cnn2-c1ccccc1',
 'CCCc1ccc2nccc(NC(=O)Nc3cccc(C(F)(F)F)n3)c2c1',
 'O=C(Cc1ccc2ccccc2c1)Nc1cc(C2CC2)[nH]n1',
 'Cc1ccc(F)c(NC(=O)Nc2ccc(-c3cncc4c3c(N)nn4C)cc2)c1',
 'NCCNS(=O)(=O)c1cccc2cnccc12',
 'CCCNC(=O)c1ccc(Nc2nc(NCC(F)(F)F)c3sccc3n2)cc1',
 'COc1ccc(C=C2SC(Nc3ccccc3)=NC2=O)cc1O',
 'COc1cccc2c1Nc1cc(Nc3ccncc3)c

## Mapping smiles to CID

In [ ]:
import requests
from urllib.parse import quote
import time

def batch_smiles_to_cids(smiles_list: list[str]) -> dict[str, list[int] | None]:
    """Map a list of SMILES strings to their PubChem CIDs."""
    results = {}
    for smiles in smiles_list:
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/{quote(smiles)}/cids/JSON"
        response = requests.get(url)
        if response.status_code == 200:
            results[smiles] = response.json()["IdentifierList"]["CID"]
        else:
            results[smiles] = None
        time.sleep(0.2)  # Respect PubChem rate limits (~5 requests/sec)
    return results

# Example
drugs = [
'O=C1c2c(c3c4ccc(O)cc4n(C4OC(CO)C(O)C(O)C4O)c3c3[nH]c4cc(O)ccc4c23)C(=O)N1NC(CO)CO'

]

cid_map = batch_smiles_to_cids(drugs)
for smi, cids in cid_map.items():
    print(f"{smi[:30]}... → CID(s): {cids}")

In [189]:
"""
Bulk SMILES → PubChem CID mapper
---------------------------------
- Threaded concurrency (concurrent.futures.ThreadPoolExecutor): zero blocking wait
- Batched requests (BATCH_SIZE SMILES per API call)
- ThreadPoolExecutor-based rate limiting (~5 req/s, respecting PubChem limits)
- Token-bucket rate limiter (REQUESTS_PER_MIN hard cap)
- Exponential backoff on transient failures
- Checkpoint saving every N drugs processed
- tqdm progress bar
- Two output JSON files:
    - success_output.json : { smiles -> [cids] }
    - failed_output.json  : { smiles -> error_reason }
"""

import json
import logging
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from urllib.parse import quote

import requests
from requests.adapters import HTTPAdapter
from tqdm import tqdm
from urllib3.util.retry import Retry

# ──────────────────────────────────────────────
# Configuration

INPUT_SMILES_FILE   = "smiles_list.txt"      # One SMILES per line
SUCCESS_OUTPUT_FILE = "cid_smile_mapping.json"
FAILED_OUTPUT_FILE  = "failed_cid_smile_mapping.json"

BATCH_SIZE          = 100   # SMILES per API request (PubChem recommended max)
CHECKPOINT_EVERY    = 10   # Save to disk every N drugs processed
MAX_WORKERS         = 4     # Max concurrent threads
REQUESTS_PER_MIN    = 100   # Hard cap on requests per minute (token-bucket)
MAX_RETRIES         = 4     # Retry attempts per request on failure
BACKOFF_BASE        = 5.0   # Exponential backoff base (seconds)
REQUEST_TIMEOUT     = 30    # Seconds before a request times out



PUBCHEM_URL = (
    "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/{smiles}/cids/JSON"
)

# ──────────────────────────────────────────────
# Logging (file only — tqdm owns stdout)
# ──────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("smiles_to_cid.log"),
    ],
)
log = logging.getLogger(__name__)


# ──────────────────────────────────────────────
# Token-bucket rate limiter
# ──────────────────────────────────────────────
class TokenBucketRateLimiter:
    """
    Thread-safe token-bucket rate limiter.

    Allows up to `rate` requests per minute, distributed evenly.
    Each call to `acquire()` blocks the calling thread just long
    enough to stay within the rate limit — no sleeping longer than
    necessary.

    Args:
        rate (int): Maximum number of requests allowed per minute.
    """

    def __init__(self, rate: int) -> None:
        self._rate      = rate                  # tokens per minute
        self._interval  = 60.0 / rate           # seconds per token
        self._lock      = threading.Lock()
        self._last_time = time.monotonic()

    def acquire(self) -> None:
        """Block until a token is available, then consume it."""
        with self._lock:
            now   = time.monotonic()
            wait  = self._last_time + self._interval - now
            if wait > 0:
                time.sleep(wait)
            self._last_time = time.monotonic()


# Single shared limiter used by all threads
_rate_limiter = TokenBucketRateLimiter(rate=REQUESTS_PER_MIN)


# ──────────────────────────────────────────────
# Per-thread requests Session
# ──────────────────────────────────────────────
_thread_local = threading.local()

def get_session() -> requests.Session:
    """Return a thread-local requests.Session (created once per thread)."""
    if not hasattr(_thread_local, "session"):
        session = requests.Session()
        retry = Retry(total=0, raise_on_status=False)
        adapter = HTTPAdapter(
            max_retries=retry,
            pool_connections=MAX_WORKERS,
            pool_maxsize=MAX_WORKERS,
        )
        session.mount("https://", adapter)
        session.mount("http://", adapter)
        _thread_local.session = session
    return _thread_local.session


# ──────────────────────────────────────────────
# Checkpoint helpers
# ──────────────────────────────────────────────
def load_checkpoint(filepath: str) -> dict:
    """Load existing results from a JSON file (resume support)."""
    p = Path(filepath)
    if p.exists():
        with open(p, "r") as f:
            data = json.load(f)
        tqdm.write(f"[resume] Loaded {len(data)} existing entries from '{filepath}'")
        return data
    return {}


def save_checkpoint(data: dict, filepath: str) -> None:
    """Atomically write results to a JSON file via tmp → rename."""
    tmp = filepath + ".tmp"
    with open(tmp, "w") as f:
        json.dump(data, f, indent=2)
    Path(tmp).replace(filepath)
    log.info(f"Checkpoint saved → '{filepath}' ({len(data)} entries)")
    tqdm.write(f"[checkpoint] Saved '{filepath}' ({len(data)} entries)")


# ──────────────────────────────────────────────
# Core fetch (single SMILES, runs in a thread)
# ──────────────────────────────────────────────
def fetch_cids(smiles: str) -> tuple[str, list[int] | None, str | None]:
    """
    Fetch CID(s) for a single SMILES string with manual exponential backoff.
    Acquires a rate-limiter token before every HTTP request.

    Returns:
        (smiles, [cids], None)        on success
        (smiles, None, error_reason)  on failure
    """
    session = get_session()
    url = PUBCHEM_URL.format(smiles=quote(smiles, safe=""))

    for attempt in range(1, MAX_RETRIES + 1):
        _rate_limiter.acquire()   # ← block here if we're over REQUESTS_PER_MIN
        try:
            resp = session.get(url, timeout=REQUEST_TIMEOUT)

            if resp.status_code == 200:
                cids = resp.json().get("IdentifierList", {}).get("CID", [])
                return smiles, cids, None

            elif resp.status_code == 404:
                # Definitive: compound not found — do not retry
                return smiles, None, "HTTP 404: compound not found"

            elif resp.status_code in (429, 503):
                wait = BACKOFF_BASE ** attempt
                log.warning(
                    f"Rate limited (HTTP {resp.status_code}) for "
                    f"'{smiles[:30]}...', attempt {attempt}/{MAX_RETRIES}. "
                    f"Waiting {wait:.1f}s"
                )
                time.sleep(wait)

            else:
                wait = BACKOFF_BASE ** attempt
                log.warning(
                    f"HTTP {resp.status_code} for '{smiles[:30]}...', "
                    f"attempt {attempt}/{MAX_RETRIES}. Waiting {wait:.1f}s"
                )
                time.sleep(wait)

        except requests.exceptions.RequestException as exc:
            wait = BACKOFF_BASE ** attempt
            log.warning(
                f"Network error for '{smiles[:30]}...': {exc}, "
                f"attempt {attempt}/{MAX_RETRIES}. Waiting {wait:.1f}s"
            )
            time.sleep(wait)

    return smiles, None, f"Failed after {MAX_RETRIES} retries"


# ──────────────────────────────────────────────
# Batch processor with checkpointing + tqdm
# ──────────────────────────────────────────────
def process_all(smiles_list: list[str]) -> None:
    # Load any prior checkpoint so we can resume interrupted runs
    success: dict[str, list[int]] = load_checkpoint(SUCCESS_OUTPUT_FILE)
    failed:  dict[str, str]       = load_checkpoint(FAILED_OUTPUT_FILE)

    # Skip already-processed SMILES
    already_done = set(success.keys()) | set(failed.keys())
    pending = [s for s in smiles_list if s not in already_done]

    tqdm.write(
        f"Total: {len(smiles_list)} | "
        f"Already done: {len(already_done)} | "
        f"Remaining: {len(pending)} | "
        f"Rate limit: {REQUESTS_PER_MIN} req/min"
    )

    if not pending:
        tqdm.write("Nothing to process. Exiting.")
        return

    processed_since_checkpoint = 0
    start_time = time.perf_counter()

    # tqdm bar — starts at already_done so the count reflects the full dataset
    progress = tqdm(
        total=len(smiles_list),
        initial=len(already_done),
        desc="Mapping SMILES → CID",
        unit="drug",
        dynamic_ncols=True,
        colour="green",
    )

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        for batch_start in range(0, len(pending), BATCH_SIZE):
            batch = pending[batch_start : batch_start + BATCH_SIZE]

            future_to_smiles = {
                executor.submit(fetch_cids, smiles): smiles
                for smiles in batch
            }

            for future in as_completed(future_to_smiles):
                smiles, cids, error = future.result()

                if error is None:
                    success[smiles] = cids
                else:
                    failed[smiles] = error
                    log.warning(f"Failed: '{smiles[:40]}' — {error}")

                processed_since_checkpoint += 1

                # Update tqdm bar postfix with live stats
                elapsed = time.perf_counter() - start_time
                rate    = (len(success) + len(failed)) / elapsed if elapsed > 0 else 0
                progress.set_postfix(
                    success=len(success),
                    failed=len(failed),
                    rate=f"{rate:.1f}/s",
                    refresh=False,
                )
                progress.update(1)

                # Checkpoint every N drugs
                if processed_since_checkpoint >= CHECKPOINT_EVERY:
                    save_checkpoint(success, SUCCESS_OUTPUT_FILE)
                    save_checkpoint(failed,  FAILED_OUTPUT_FILE)
                    processed_since_checkpoint = 0

    progress.close()

    # Final save
    save_checkpoint(success, SUCCESS_OUTPUT_FILE)
    save_checkpoint(failed,  FAILED_OUTPUT_FILE)

    elapsed = time.perf_counter() - start_time
    tqdm.write(
        f"\nDone! Total: {len(smiles_list)} | "
        f"Success: {len(success)} | Failed: {len(failed)} | "
        f"Time: {elapsed:.1f}s"
    )


# ──────────────────────────────────────────────
# Entry point
# ──────────────────────────────────────────────
def main() -> None:

    tqdm.write(f"Loaded {len(kiba_drugs[:50])} SMILES from '{INPUT_SMILES_FILE}'")
    process_all(kiba_drugs[:50])


if __name__ == "__main__":
    main()

Loaded 50 SMILES from 'smiles_list.txt'
Total: 50 | Already done: 0 | Remaining: 50 | Rate limit: 100 req/min


Mapping SMILES → CID:  20%|██        | 10/50 [00:06<00:24,  1.66drug/s, failed=0, rate=1.5/s, success=10]

[checkpoint] Saved 'cid_smile_mapping.json' (10 entries)
[checkpoint] Saved 'failed_cid_smile_mapping.json' (0 entries)


Mapping SMILES → CID:  40%|████      | 20/50 [00:12<00:17,  1.68drug/s, failed=0, rate=1.6/s, success=20]

[checkpoint] Saved 'cid_smile_mapping.json' (20 entries)
[checkpoint] Saved 'failed_cid_smile_mapping.json' (0 entries)


...

[checkpoint] Saved 'cid_smile_mapping.json' (30 entries)
[checkpoint] Saved 'failed_cid_smile_mapping.json' (0 entries)


Mapping SMILES → CID:  80%|████████  | 40/50 [00:24<00:06,  1.64drug/s, failed=0, rate=1.6/s, success=40]

[checkpoint] Saved 'cid_smile_mapping.json' (40 entries)
[checkpoint] Saved 'failed_cid_smile_mapping.json' (0 entries)


Mapping SMILES → CID: 100%|██████████| 50/50 [00:30<00:00,  1.64drug/s, failed=0, rate=1.6/s, success=50]
2026-04-09 19:08:05,639 [INFO] Checkpoint saved → 'cid_smile_mapping.json' (50 entries)
2026-04-09 19:08:05,640 [INFO] Checkpoint saved → 'failed_cid_smile_mapping.json' (0 entries)


[checkpoint] Saved 'cid_smile_mapping.json' (50 entries)
[checkpoint] Saved 'failed_cid_smile_mapping.json' (0 entries)
[checkpoint] Saved 'cid_smile_mapping.json' (50 entries)
[checkpoint] Saved 'failed_cid_smile_mapping.json' (0 entries)

Done! Total: 50 | Success: 50 | Failed: 0 | Time: 30.6s


In [ ]:
"""
Bulk SMILES → PubChem CID mapper
---------------------------------
- Threaded concurrency (concurrent.futures.ThreadPoolExecutor): zero blocking wait
- Batched requests (BATCH_SIZE SMILES per API call)
- ThreadPoolExecutor-based rate limiting (~5 req/s, respecting PubChem limits)
- Exponential backoff on transient failures
- Checkpoint saving every N drugs processed
- Two output JSON files:
    - success_output.json : { smiles -> [cids] }
    - failed_output.json  : { smiles -> error_reason }
"""

import json
import logging
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from urllib.parse import quote

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ──────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────

INPUT_SMILES_FILE   = "smiles_list.txt"      # One SMILES per line
SUCCESS_OUTPUT_FILE = "cid_smile_mapping.json"
FAILED_OUTPUT_FILE  = "failed_cid_smile_mapping.json"

BATCH_SIZE        = 100   # SMILES per API request (PubChem recommended max)
CHECKPOINT_EVERY  = 10   # Save to disk every N drugs processed
MAX_WORKERS       = 4     # Max concurrent threads (≤5 req/s for PubChem)
MAX_RETRIES       = 4     # Retry attempts per request on failure
BACKOFF_BASE      = 10   # Exponential backoff base (seconds)
REQUEST_TIMEOUT   = 30    # Seconds before a request times out


# INPUT_SMILES_FILE   = "smiles_list.txt"      # One SMILES per line
# SUCCESS_OUTPUT_FILE = "cid_smile_mapping.json"
# FAILED_OUTPUT_FILE  = "failed_cid_smile_mapping.json"

# BATCH_SIZE          = 100    # SMILES per API request (PubChem recommended max)
# CHECKPOINT_EVERY    = 10    # Save to disk every N drugs processed
# MAX_CONCURRENT      = 2      # Max concurrent requests (≤5 req/s for PubChem)
# MAX_RETRIES         = 4      # Retry attempts per batch on failure
# BACKOFF_BASE        = 2.0    # Exponential backoff base (seconds)


PUBCHEM_URL = (
    "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/{smiles}/cids/JSON"
)

# ──────────────────────────────────────────────
# Logging
# ──────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("smiles_to_cid.log"),
    ],
)
log = logging.getLogger(__name__)


# ──────────────────────────────────────────────
# Per-thread requests Session (with built-in retry)
# ──────────────────────────────────────────────
def make_session() -> requests.Session:
    """
    Create a requests.Session with a mounted HTTPAdapter.
    Each thread gets its own session (thread-local).
    Built-in urllib3 Retry handles connection-level errors;
    application-level retries (429/503) are handled manually.
    """
    session = requests.Session()
    retry = Retry(
        total=0,               # We handle retries manually
        raise_on_status=False,
    )
    adapter = HTTPAdapter(
        max_retries=retry,
        pool_connections=MAX_WORKERS,
        pool_maxsize=MAX_WORKERS,
    )
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


# Thread-local storage so each thread reuses its own Session
import threading
_thread_local = threading.local()

def get_session() -> requests.Session:
    if not hasattr(_thread_local, "session"):
        _thread_local.session = make_session()
    return _thread_local.session


# ──────────────────────────────────────────────
# Checkpoint helpers
# ──────────────────────────────────────────────
def load_checkpoint(filepath: str) -> dict:
    """Load existing results from a JSON file (resume support)."""
    p = Path(filepath)
    if p.exists():
        with open(p, "r") as f:
            data = json.load(f)
        log.info(f"Loaded {len(data)} existing entries from '{filepath}'")
        return data
    return {}


def save_checkpoint(data: dict, filepath: str) -> None:
    """Atomically write results to a JSON file via tmp → rename."""
    tmp = filepath + ".tmp"
    with open(tmp, "w") as f:
        json.dump(data, f, indent=2)
    Path(tmp).replace(filepath)  # atomic rename — never a half-written file
    log.info(f"Checkpoint saved → '{filepath}' ({len(data)} entries)")


# ──────────────────────────────────────────────
# Core fetch (single SMILES, runs in a thread)
# ──────────────────────────────────────────────
def fetch_cids(smiles: str) -> tuple[str, list[int] | None, str | None]:
    """
    Fetch CID(s) for a single SMILES string with manual exponential backoff.

    Returns:
        (smiles, [cids], None)        on success
        (smiles, None, error_reason)  on failure
    """
    session = get_session()
    url = PUBCHEM_URL.format(smiles=quote(smiles, safe=""))

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.get(url, timeout=REQUEST_TIMEOUT)

            if resp.status_code == 200:
                cids = resp.json().get("IdentifierList", {}).get("CID", [])
                return smiles, cids, None

            elif resp.status_code == 404:
                # Definitive: compound not found — do not retry
                return smiles, None, "HTTP 404: compound not found"

            elif resp.status_code in (429, 503):
                wait = BACKOFF_BASE ** attempt
                log.warning(
                    f"Rate limited (HTTP {resp.status_code}) for "
                    f"'{smiles[:30]}...', attempt {attempt}/{MAX_RETRIES}. "
                    f"Waiting {wait:.1f}s"
                )
                time.sleep(wait)

            else:
                wait = BACKOFF_BASE ** attempt
                log.warning(
                    f"HTTP {resp.status_code} for '{smiles[:30]}...', "
                    f"attempt {attempt}/{MAX_RETRIES}. Waiting {wait:.1f}s"
                )
                time.sleep(wait)

        except requests.exceptions.RequestException as exc:
            wait = BACKOFF_BASE ** attempt
            log.warning(
                f"Network error for '{smiles[:30]}...': {exc}, "
                f"attempt {attempt}/{MAX_RETRIES}. Waiting {wait:.1f}s"
            )
            time.sleep(wait)

    return smiles, None, f"Failed after {MAX_RETRIES} retries"


# ──────────────────────────────────────────────
# Batch processor with checkpointing
# ──────────────────────────────────────────────
def process_all(smiles_list: list[str]) -> None:
    # Load any prior checkpoint so we can resume interrupted runs
    success: dict[str, list[int]] = load_checkpoint(SUCCESS_OUTPUT_FILE)
    failed:  dict[str, str]       = load_checkpoint(FAILED_OUTPUT_FILE)

    # Skip already-processed SMILES
    already_done = set(success.keys()) | set(failed.keys())
    pending = [s for s in smiles_list if s not in already_done]
    log.info(
        f"Total SMILES: {len(smiles_list)} | "
        f"Already processed: {len(already_done)} | "
        f"Remaining: {len(pending)}"
    )

    if not pending:
        log.info("Nothing to process. Exiting.")
        return

    processed_since_checkpoint = 0
    total_processed = len(already_done)
    start_time = time.perf_counter()

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Submit in batches to avoid saturating the queue with 20k futures at once
        for batch_start in range(0, len(pending), BATCH_SIZE):
            batch = pending[batch_start : batch_start + BATCH_SIZE]

            # Submit all tasks in the batch and collect as they complete
            future_to_smiles = {
                executor.submit(fetch_cids, smiles): smiles
                for smiles in batch
            }

            for future in as_completed(future_to_smiles):
                smiles, cids, error = future.result()
                if error is None:
                    success[smiles] = cids
                else:
                    failed[smiles] = error

                processed_since_checkpoint += 1
                total_processed            += 1

                # Progress log every 100 drugs
                if total_processed % 100 == 0:
                    elapsed = time.perf_counter() - start_time
                    rate    = total_processed / elapsed if elapsed > 0 else 0
                    log.info(
                        f"Progress: {total_processed}/{len(smiles_list)} drugs "
                        f"({100 * total_processed / len(smiles_list):.1f}%) | "
                        f"Speed: {rate:.1f} drugs/s | "
                        f"Success: {len(success)} | Failed: {len(failed)}"
                    )

                # Checkpoint every N drugs
                if processed_since_checkpoint >= CHECKPOINT_EVERY:
                    save_checkpoint(success, SUCCESS_OUTPUT_FILE)
                    save_checkpoint(failed,  FAILED_OUTPUT_FILE)
                    processed_since_checkpoint = 0

    # Final save
    save_checkpoint(success, SUCCESS_OUTPUT_FILE)
    save_checkpoint(failed,  FAILED_OUTPUT_FILE)

    elapsed = time.perf_counter() - start_time
    log.info(
        f"Done! Total: {len(smiles_list)} | "
        f"Success: {len(success)} | Failed: {len(failed)} | "
        f"Time: {elapsed:.1f}s"
    )


# ──────────────────────────────────────────────
# Entry point
# ──────────────────────────────────────────────
def main() -> None:
    log.info(f"Loaded {len(kiba_drugs[:50])} SMILES from '{INPUT_SMILES_FILE}'")
    process_all(kiba_drugs[:50])


if __name__ == "__main__":
    main()

2026-04-09 19:01:02,042 [INFO] Loaded 50 SMILES from 'smiles_list.txt'
2026-04-09 19:01:02,044 [INFO] Total SMILES: 50 | Already processed: 0 | Remaining: 50
2026-04-09 19:01:04,582 [INFO] Checkpoint saved → 'cid_smile_mapping.json' (10 entries)
2026-04-09 19:01:04,584 [INFO] Checkpoint saved → 'failed_cid_smile_mapping.json' (0 entries)
2026-04-09 19:01:05,042 [WARNING] Rate limited (HTTP 503) for 'Cc1ccc(F)c(NC(=O)Nc2ccc(-c3cnc...', attempt 1/4. Waiting 2.0s
2026-04-09 19:01:05,108 [WARNING] Rate limited (HTTP 503) for 'NCCNS(=O)(=O)c1cccc2cnccc12...', attempt 1/4. Waiting 2.0s
2026-04-09 19:01:07,299 [INFO] Checkpoint saved → 'cid_smile_mapping.json' (20 entries)
2026-04-09 19:01:07,300 [INFO] Checkpoint saved → 'failed_cid_smile_mapping.json' (0 entries)
2026-04-09 19:01:08,852 [INFO] Checkpoint saved → 'cid_smile_mapping.json' (30 entries)
2026-04-09 19:01:08,853 [INFO] Checkpoint saved → 'failed_cid_smile_mapping.json' (0 entries)
2026-04-09 19:01:10,275 [INFO] Checkpoint saved →

## InchI Key

In [1]:
import time
import requests
import pandas as pd
from tqdm import tqdm


In [2]:
data_path = Path('/DATA/DATANAS2/rhh25/Cellhit/data/')

df_drugcombs = pd.read_csv(f'{data_path}/metadata/drugcombs_scored.csv')
df_drugcombs = df_drugcombs.dropna(subset=['Drug1', 'Drug2'] )
unique_drugs =  set(df_drugcombs.Drug1.unique().tolist()).union(
        set(df_drugcombs.Drug2.unique().tolist())
    )


NameError: name 'Path' is not defined

In [40]:
INPUT_CSV      = "drugcomb_data.csv"   # path to your DrugComb CSV
DRUG_NAME_COL  = "drug_row"            # column that holds drug names
OUTPUT_CSV     = "drugcomb_inchikeys.csv"
 
PUBCHEM_URL    = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
REQUEST_DELAY  = 0.22         